[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/herrrickshaw/global-stock-screener/blob/main/notebooks/01_data_layer.ipynb)


# Checkpoint 01 — Central LFS Data Layer
Loads the committed **LFS data** (5y LTM per market) and exercises the storage / linkage / serving big-functions.
- `market_memory` — LTM(5y) ↔ STM(1y) two-tier memory
- `datalink` — manifest, memoized loads, incremental LMDB store
- `serving_layer` — materialised feature views + CDC deltas
- `export_company_list` — the downloadable master list


In [ ]:
# Setup — clone the repo + pull the central LFS data (idempotent; safe to re-run)
!pip install -q lmdb pyarrow scikit-learn nsepython yfinance vaderSentiment openpyxl
!git lfs install
%cd /content
![ -d global-stock-screener ] || git clone --depth 1 https://github.com/herrrickshaw/global-stock-screener.git
%cd /content/global-stock-screener
!git lfs pull
import os; os.environ['BHAV_CACHE'] = '/content/cache'
import market_memory as mm, serving_layer as sl
mm.ensure_stm_seeds(verbose=False)   # derive 1y STM from the committed 5y LTM
sl.refresh(['IN', 'US'])             # build serving views used below


### Central data manifest (what LFS shipped)


In [ ]:
import datalink; man = datalink.build_manifest(verbose=False)
import pandas as pd; pd.DataFrame(man['markets']).head(20)


### Two-tier memory: LTM (5y batch) → STM (1y speed)


In [ ]:
import market_memory as mm; print(mm.status().to_string(index=False))


### Serving views + CDC (denormalised, precomputed)


In [ ]:
import serving_layer as sl; sl.refresh(['IN','US'])
sl.serving('IN')[['Symbol','Close','RSI14','Ret252','Liquidity']].head()


### Master company list (all analysed companies)


In [ ]:
import export_company_list as e; df=e.build(); print(len(df),'companies'); df.head()
